# RC Circuits and the Leaky Integrate-and-Fire Neuron

Welcome! In this exercise, you’ll simulate a simple RC circuit, which behaves similarly to a neuron’s membrane.

You’ll learn how voltage changes over time in response to an input current and how this forms the basis of the Leaky Integrate-and-Fire (LIF) neuron model.

The governing equation of an RC circuit is:

tau_m * dV/dt = -(V - V_rest) + R * I(t)

where:
- V is the membrane voltage (mV)
- R is the resistance (MOhm)
- C is the capacitance (nF)
- I(t) is the input current (nA)
- tau_m = R * C is the membrane time constant (ms)

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Define simulation parameters

tStop = 200       # total duration (ms)
dt = 0.1          # time step (ms)
time = np.arange(0, tStop + dt, dt)

# Circuit parameters
R = 100           # Resistance (MOhm)
C = 0.05          # Capacitance (nF)
V_rest = -65        # Resting potential (mV)
tau_m = R * C     # Membrane time constant (ms)

# Input current
I_amp = 0.2       # Current amplitude (nA)
I_start = 50      # Start time (ms)
I_duration = 100  # Duration (ms)

## Step 1: Euler Integration (Student Task)

We’ll use Euler’s method to numerically integrate the differential equation:

V[t+1] = V[t] + (dt / C) * ( - (V[t] - V_rest) / R + I[t] )

Your task: complete the missing line of code below that updates the membrane potential.

In [ ]:
# Step 1: Integration loop
# Initialize membrane potential
V = np.zeros_like(time)
V[0] = V_rest

for t in range(0, len(time)-1):
    # TODO: Implement Euler integration rule here
    # Stimulus current
        IStim = 0
        if t>=int(I_start/dt) and t<int((I_start+I_duration)/dt):
            IStim = I_amp # in nA
        
        # Compute change of v
        V[t+1] = V[t]+((-(V[t]-V_rest)/R+IStim)/C)*dt # in (nA/nF)*ms==mV


In [ ]:
# Step 2: Plot results
# TODO: Plot the input and cell response here
fig = plt.figure()
ax = fig.add_subplot(111)
ax.grid()
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.plot(time,V,'b',label='Response')
plt.xlim(0,tStop)
plt.ylim(min(V)-55,max(V)+55)
rect = plt.Rectangle((I_start, min(V)-55), I_duration, 20, fill=True, color='red', label='Input Timing')
ax.add_patch(rect)
ax.legend()
plt.xlabel('Time (ms)')
plt.ylabel('Membrane voltage (mV)')
plt.show()

## Step 3: Exploration Questions

1. What happens to the membrane potential if you double the resistance R?
2. How does increasing C change the charging/discharging rate?
3. Try changing the time step dt. What happens if it’s too large?

In [ ]:
## Step 4: Add Spiking Behavior with a Refractory Period

# TODO: Modify the model from Step 2 so that it becomes a Leaky Integrate-and-Fire model here
# TODO: In the plot indicate action potentials by plotting vertical lines from V_th to 100 mV

# Parameters
V_th = -55      # Spike threshold (mV)
V_reset = -70   # Reset potential (mV)
t_ref = 3      # Refractory period (ms)


# Initialize membrane potential
V = np.zeros_like(time)
V[0] = V_rest

AP_timings = []
ref_counter = 0
for t in range(0, len(time)-1):
    # Stimulus current
    IStim = 0
    if t>=int(I_start/dt) and t<int((I_start+I_duration)/dt):
        IStim = I_amp # in nA

    # If not in refractory period
    if ref_counter==0:
        # Compute change of v
        V[t+1] = V[t]+((-(V[t]-V_rest)/R+IStim)/C)*dt # in (nA/nF)*ms==mV

        # Check for AP, threshold crossing
        if V[t+1]>=V_th:
            V[t+1] = V_reset # reset Vm
            AP_timings.append(t) # record AP timing
            ref_counter = int(t_ref/dt) # add counter for refractory period
    # If in refractory period
    else: 
        V[t+1] = V[t] # keep Vm the same
        ref_counter-=1 # decrease counter for refractory period

fig = plt.figure()
ax = fig.add_subplot(111)
ax.grid()
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.plot(time,V,'b',label='Response')
plt.vlines(x=np.array(AP_timings)*dt, ymin=V_th, ymax=V_th+100, color='b')
plt.xlim(0,tStop)
plt.ylim(min(V)-55,max(V)+100+10)
rect = plt.Rectangle((I_start, min(V)-55), I_duration, 20, fill=True, color='red', label='Input Timing')
ax.add_patch(rect)
ax.legend()
plt.xlabel('Time (ms)')
plt.ylabel('Membrane voltage (mV)')
plt.show()
